# 🧬 Urban DNA Sequencer - Phase 1
## Extract, Analyze & Cluster Building Morphology

**Location:** Kigali, Rwanda (3km radius)  
**Data Source:** Google Open Buildings V3 (2023)  
**Goal:** Identify urban typologies using building geometry

## 📦 Step 1: Install & Import

In [ ]:
!pip install geemap folium -q

In [ ]:
import ee
import geemap
import folium
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

print('✅ Libraries loaded')

## 🔐 Step 2: Authenticate Earth Engine

In [ ]:
try:
    ee.Initialize(project='rwanda-urban-dna')
    print('✅ Already authenticated')
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='rwanda-urban-dna')
    print('✅ Authentication complete')

## 📍 Step 3: Define ROI (Kigali)

In [ ]:
kigali_center = ee.Geometry.Point([30.0619, -1.9441])
roi = kigali_center.buffer(3000)

print(f'✅ ROI defined: 3km radius around Kigali center')

## 🏢 Step 4: Load Buildings (Confidence > 75%)

In [ ]:
buildings = ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons') \
    .filterBounds(roi) \
    .filter(ee.Filter.gt('confidence', 0.75))

count = buildings.size().getInfo()
print(f'✅ Loaded {count:,} buildings')

## 🧬 Step 5: Calculate Building DNA (Geometry Features)

In [ ]:
def add_dna(feature):
    geom = feature.geometry()
    area = geom.area()
    perimeter = geom.perimeter()
    shape_index = perimeter.pow(2).divide(area)
    
    return feature.set({
        'area': area,
        'perimeter': perimeter,
        'shape_index': shape_index
    })

dna_buildings = buildings.map(add_dna)
print('✅ Geometry features calculated')

## 📏 Step 6: Calculate Nearest Neighbor Distance

In [ ]:
max_dist_filter = ee.Filter.withinDistance(
    distance=50,
    leftField='.geo',
    rightField='.geo',
    maxError=1
)

join = ee.Join.saveBest(
    matchKey='nearest',
    measureKey='nn_dist'
)

buildings_with_nn = join.apply(dna_buildings, dna_buildings, max_dist_filter)

def flatten_nn(feature):
    neighbor = feature.get('nearest')
    dist = ee.Algorithms.If(
        neighbor,
        feature.get('nn_dist'),
        99
    )
    return feature.set({'nn_dist': dist})

final_buildings = buildings_with_nn.map(flatten_nn)
print('✅ Nearest neighbor distances calculated')

## 💾 Step 7: Download to Pandas DataFrame

In [ ]:
export_fc = final_buildings.select(
    ['area', 'perimeter', 'shape_index', 'nn_dist', 'confidence'],
    retainGeometry=True
)

gdf = geemap.ee_to_geopandas(export_fc)

print(f'✅ Downloaded {len(gdf):,} buildings')
print(f'Columns: {list(gdf.columns)}')
gdf.head()

## 📊 Step 8: Exploratory Data Analysis

In [ ]:
print('=== BUILDING DNA STATISTICS ===')
print(gdf[['area', 'perimeter', 'shape_index', 'nn_dist']].describe())

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

gdf['area'].hist(bins=50, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Building Area (m²)')
axes[0,0].set_xlabel('Area')

gdf['shape_index'].hist(bins=50, ax=axes[0,1], color='coral')
axes[0,1].set_title('Shape Index (Compactness)')
axes[0,1].set_xlabel('Shape Index')

gdf['nn_dist'].hist(bins=50, ax=axes[1,0], color='green')
axes[1,0].set_title('Nearest Neighbor Distance (m)')
axes[1,0].set_xlabel('Distance')

gdf['confidence'].hist(bins=20, ax=axes[1,1], color='purple')
axes[1,1].set_title('Detection Confidence')
axes[1,1].set_xlabel('Confidence')

plt.tight_layout()
plt.show()

## 🎯 Step 9: K-Means Clustering

In [ ]:
features = ['area', 'shape_index', 'nn_dist']
X = gdf[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertias = []
K_range = range(2, 8)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.grid(True)
plt.show()

print('👆 Look for the "elbow" - typically K=3 or K=4')

In [ ]:
optimal_k = 4

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
gdf['cluster'] = kmeans.fit_predict(X_scaled)

print(f'✅ Clustered into {optimal_k} urban typologies')
print('\nCluster distribution:')
print(gdf['cluster'].value_counts().sort_index())

## 🏷️ Step 10: Interpret Clusters

In [ ]:
cluster_stats = gdf.groupby('cluster')[features].mean()
cluster_stats['count'] = gdf.groupby('cluster').size()

print('=== CLUSTER CHARACTERISTICS ===')
print(cluster_stats)

print('\n=== INTERPRETATION ===')
for i in range(optimal_k):
    stats = cluster_stats.loc[i]
    
    size = 'Small' if stats['area'] < 50 else 'Medium' if stats['area'] < 150 else 'Large'
    density = 'Dense' if stats['nn_dist'] < 5 else 'Moderate' if stats['nn_dist'] < 15 else 'Sparse'
    shape = 'Regular' if stats['shape_index'] < 20 else 'Irregular'
    
    print(f"Cluster {i}: {size} buildings, {density} spacing, {shape} shapes ({int(stats['count'])} buildings)")

## 🗺️ Step 11: Visualize Clusters on Map

In [ ]:
colors = ['red', 'blue', 'green', 'orange', 'purple', 'yellow']

Map = folium.Map(location=[-1.9441, 30.0619], zoom_start=14)

for i in range(optimal_k):
    cluster_gdf = gdf[gdf['cluster'] == i]
    folium.GeoJson(
        cluster_gdf,
        name=f'Cluster {i}',
        style_function=lambda x, color=colors[i]: {
            'fillColor': color,
            'color': color,
            'weight': 1,
            'fillOpacity': 0.6
        }
    ).add_to(Map)

folium.LayerControl().add_to(Map)
print(f'✅ Visualized {optimal_k} clusters on map')
Map

## 💾 Step 12: Export Results

In [ ]:
output_file = f'kigali_urban_dna_{datetime.now().strftime("%Y%m%d")}.geojson'
gdf.to_file(output_file, driver='GeoJSON')

print(f'✅ Exported to {output_file}')
print(f'File size: {len(gdf):,} buildings')

cluster_stats.to_csv('cluster_summary.csv')
print('✅ Cluster summary saved to cluster_summary.csv')

## 📈 Step 13: Final Summary

In [ ]:
print('=' * 50)
print('🧬 URBAN DNA SEQUENCER - RESULTS')
print('=' * 50)
print(f'Location: Kigali, Rwanda (3km radius)')
print(f'Total buildings analyzed: {len(gdf):,}')
print(f'Urban typologies identified: {optimal_k}')
print(f'\nMean building area: {gdf["area"].mean():.1f} m²')
print(f'Mean shape index: {gdf["shape_index"].mean():.1f}')
print(f'Mean NN distance: {gdf["nn_dist"].mean():.1f} m')
print(f'\nData exported: {output_file}')
print('=' * 50)
print('✅ Analysis complete!')